In [1]:
import sys
from pathlib import Path

from sympy import fraction

sys.path.append(str(Path.cwd().parent))
import importlib
if 'src.data_processing' in sys.modules:
    import src.data_processing
    importlib.reload(src.data_processing)

from data.download_kaggle import download_unicon
from src.data_processing import merge_all_datasets, train_test_split_temporal, save_processed_data, process_data_pipeline



In [2]:
# Wczytaj dane

datasets = download_unicon()

df_weather = datasets['weather']
df_bc = datasets['consumption']
df_events = datasets['events']
df_cal = datasets['calendar']
df_building_meta = datasets['building_meta']
df_campus_meta = datasets['campus_meta']


✓ dane z cache (data/raw)
✓ Wczytano 6 plików CSV
  - weather: 7,396,520 wierszy, 8 kolumn
  - consumption: 8,095,524 wierszy, 4 kolumn
  - events: 106 wierszy, 4 kolumn
  - calendar: 2,312 wierszy, 4 kolumn
  - building_meta: 64 wierszy, 7 kolumn
  - campus_meta: 5 wierszy, 3 kolumn


In [3]:
# Merge datasetów
df_merged = merge_all_datasets(
    df_bc, df_weather, df_events, df_cal, df_building_meta, df_campus_meta
)

Łączenie datasetów...
  Consumption: (8095524, 4)
  + Building meta: (8095524, 11)
  + Campus meta: (8095524, 14)
  + Weather: (8095524, 20)
  + Calendar: (8095524, 24)
✓ Merge zakończony. Wymiary: (8095524, 21)
  Kolumny (21): ['campus_id', 'meter_id', 'timestamp', 'consumption', 'campus_id_building', 'built_year', 'category', 'gross_floor_area', 'room_area', 'capacity', 'name', 'capacity_campus', 'apparent_temperature', 'air_temperature', 'dew_point_temperature', 'relative_humidity', 'wind_speed', 'wind_direction', 'is_holiday', 'is_semester', 'is_exam']


In [4]:
# Sprawdzenie połączonych danych
print("\nPierwsze wiersze:")
print(df_merged.head())
print(f"\nWymiary: {df_merged.shape}")
print(f"\nKolumny: {list(df_merged.columns)}")



Pierwsze wiersze:
   campus_id  meter_id           timestamp  consumption  campus_id_building  \
0          1         1 2019-03-29 01:15:00        0.011                   1   
1          1         1 2019-03-29 01:30:00        0.021                   1   
2          1         1 2019-03-29 01:45:00        0.038                   1   
3          1         1 2019-03-29 02:00:00        0.871                   1   
4          1         1 2019-03-29 02:15:00        0.873                   1   

   built_year category  gross_floor_area  room_area  capacity  ...  \
0         NaN    other               NaN        NaN       NaN  ...   
1         NaN    other               NaN        NaN       NaN  ...   
2         NaN    other               NaN        NaN       NaN  ...   
3         NaN    other               NaN        NaN       NaN  ...   
4         NaN    other               NaN        NaN       NaN  ...   

  capacity_campus  apparent_temperature  air_temperature  \
0           26000        

In [5]:
# Braki danych
print("\nBraki danych:")
missing = df_merged.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

# Temporal split (80% train, 20% test)
train_df, test_df = train_test_split_temporal(df_merged, test_size=0.2)

# Zapisz dane
save_processed_data(df_merged, "data/processed/merged_data.csv")
save_processed_data(train_df, "data/processed/train_data.csv")
save_processed_data(test_df, "data/processed/test_data.csv")




Braki danych:
room_area                2551802
capacity                 2551802
wind_direction           1512199
wind_speed               1481629
apparent_temperature      744202
air_temperature           744202
dew_point_temperature     744202
relative_humidity         744202
built_year                528746
gross_floor_area          235433
dtype: int64

Temporal split:
  Train: 6,476,419 wierszy (2018-01-01 00:15:00 to 2021-06-01 15:00:00)
  Test:  1,619,105 wierszy (2021-06-01 15:00:00 to 2022-04-30 23:45:00)
✓ Dane zapisane do: C:\Users\aiszk\PycharmProjects\PythonProject\data\processed\merged_data.csv
✓ Dane zapisane do: C:\Users\aiszk\PycharmProjects\PythonProject\data\processed\train_data.csv
✓ Dane zapisane do: C:\Users\aiszk\PycharmProjects\PythonProject\data\processed\test_data.csv


In [6]:
df_merged.columns

Index(['campus_id', 'meter_id', 'timestamp', 'consumption',
       'campus_id_building', 'built_year', 'category', 'gross_floor_area',
       'room_area', 'capacity', 'name', 'capacity_campus',
       'apparent_temperature', 'air_temperature', 'dew_point_temperature',
       'relative_humidity', 'wind_speed', 'wind_direction', 'is_holiday',
       'is_semester', 'is_exam'],
      dtype='object')

In [7]:
merged_sample = df_merged.head(50000)
#save_processed_data(merged_sample, "data/processed/merged_sample.csv")